In [59]:
from typing import Annotated,TypedDict
from langgraph.graph import START,StateGraph,END
from langchain_groq import ChatGroq

In [60]:
from dotenv import load_dotenv
load_dotenv()

True

In [61]:
class SubState(TypedDict):
    input_text: str
    translated_text: str

In [62]:
subgraph_llm=ChatGroq(model="llama-3.3-70b-versatile")

In [63]:
def translate(state:SubState):
    prompt = f"""
Translate the following text to Kannada.
Keep it natural and clear. Do not add extra content.

Text:
{state["input_text"]}
""".strip()
    
    response=subgraph_llm.invoke(prompt).content
    return {'translated_text': response}

In [64]:
subgraph_builder=StateGraph(SubState)

subgraph_builder.add_node('translate',translate)
subgraph_builder.add_edge(START,'translate')
subgraph_builder.add_edge('translate',END)

subgraph = subgraph_builder.compile()

In [65]:
class ParentState(TypedDict):
    query:str
    eng_output:str
    kannada_output:str

In [66]:
parent_llm=ChatGroq(model="llama-3.3-70b-versatile")

In [67]:
def chat(state:ParentState):
    answer = parent_llm.invoke(f"You are a helpful assistant. Answer clearly.\n\nQuestion: {state['query']}").content
    return {'eng_output': answer}

In [68]:
def translate_ans(state:ParentState):
    response=subgraph.invoke({'input_text':state['eng_output']})
    return {'kannada_output':response["translated_text"]}

In [69]:
parent_builder=StateGraph(ParentState)

parent_builder.add_node("chat",chat)
parent_builder.add_node("translate",translate_ans)
parent_builder.add_edge(START,"chat")
parent_builder.add_edge("chat","translate")
parent_builder.add_edge("translate",END)

parentgraph=parent_builder.compile()

In [70]:
parentgraph.invoke({"query":"how to get a job in ai as a fresher"})

{'query': 'how to get a job in ai as a fresher',
 'eng_output': 'As a fresher, getting a job in AI can be challenging, but with the right approach, you can increase your chances of success. Here\'s a step-by-step guide to help you get started:\n\n1. **Build a strong foundation in math and programming**:\n\t* Focus on linear algebra, calculus, probability, and statistics.\n\t* Learn programming languages like Python, R, or Julia.\n\t* Familiarize yourself with popular AI frameworks like TensorFlow, PyTorch, or Keras.\n2. **Gain relevant skills and knowledge**:\n\t* Learn machine learning algorithms, deep learning, and natural language processing.\n\t* Study computer vision, robotics, or other AI-related fields that interest you.\n\t* Take online courses or attend workshops to stay updated on the latest AI trends.\n3. **Get familiar with AI tools and technologies**:\n\t* Learn about popular AI libraries and frameworks like scikit-learn, OpenCV, or NLTK.\n\t* Experiment with AI-powered to